# Do Historical Rate Betas Predict Future FOMC Reactions?

This notebook validates the economic target before testing any embedding. A beta estimated from the preceding five years is used only to predict FOMC reactions in the following quarter. If it cannot predict later reactions better than shuffled labels, embedding recovery of that beta would not be economically meaningful.

## 1. Setup and input gates

No WRDS connection is required. The completed event panel and rolling targets are read from local checkpoints.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

def locate_root():
    start=Path.cwd().resolve()
    for candidate in [start,*start.parents]:
        if (candidate/'economic_shock_sensitivity').exists() and (candidate/'full_replication_80q').exists(): return candidate
    raise FileNotFoundError('Could not locate Asset_Embeddings root.')

ROOT=locate_root(); MODULE=ROOT/'economic_shock_sensitivity'; DATA=MODULE/'data'/'processed'
RESULTS=MODULE/'results'/'future_target_validation'; RESULTS.mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(MODULE/'src'))
import future_target_validation as ftv
acquisition_gate=pd.read_parquet(MODULE/'results'/'event_return_acquisition'/'acquisition_gate.parquet')
target_gate=pd.read_parquet(MODULE/'results'/'rolling_rate_targets'/'target_construction_gate.parquet')
display(acquisition_gate); display(target_gate)
assert acquisition_gate.passed.all() and target_gate.passed.all()


## 2. Assemble strictly forward event panels

For an event in quarter q, the predictor is the five-year beta frozen at the start of q. The primary label is the winsorized beta. Shrunken betas are not used because their estimated signal variance collapsed some quarterly cross-sections to a constant.

In [ ]:
targets=pd.read_parquet(DATA/'rolling_rate_targets'/'all_rolling_rate_targets.parquet')
events=pd.read_parquet(DATA/'fomc_event_returns'/'firm_fomc_event_returns_2005_2024.parquet')
events['event_date']=pd.to_datetime(events.event_date)
events=events[events.event_date.ge('2010-01-01')].copy()
events['anchor_quarter']=events.event_date.dt.to_period('Q').astype(str).str.lower()
specifications={'stmt_ar00':{'outcome':'abnormal_return_00','shock':'shock_stmt'},'stmt_ar01':{'outcome':'abnormal_return_01','shock':'shock_stmt'},'me_ar00':{'outcome':'abnormal_return_00','shock':'shock_me'}}
panels={}
for name,definition in specifications.items():
    label=targets[(targets.specification.eq(name))&targets.eligible][['anchor_quarter','permco','beta_raw','beta_winsorized','beta_standard_error','reliability','firm_split']]
    panel=events.merge(label,on=['anchor_quarter','permco'],how='inner',validate='many_to_one')
    panel['shock_value']=panel[definition['shock']]
    panel['prediction']=panel.beta_winsorized*panel.shock_value
    panel['time_sample']=np.select([panel.event_date.dt.year.le(2016),panel.event_date.dt.year.le(2020)],['development','validation'],default='final_test')
    panels[name]=panel
assembly=pd.DataFrame([{'specification':name,'rows':len(panel),'events':panel.event_date.nunique(),'firms':panel.permco.nunique(),'first_event':panel.event_date.min(),'last_event':panel.event_date.max()} for name,panel in panels.items()])
assembly.to_parquet(RESULTS/'forward_panel_assembly.parquet',index=False)
display(assembly)


## 3. Timing audit

The start of each event quarter is reconstructed. Every evaluated event must occur on or after that date, while the target-estimation window ended immediately before it.

In [ ]:
timing=[]
for name,panel in panels.items():
    anchor_start=pd.PeriodIndex(panel.anchor_quarter.str.upper(),freq='Q').start_time
    violations=int((panel.event_date.to_numpy()<anchor_start.to_numpy()).sum())
    timing.append({'specification':name,'rows':len(panel),'events':panel.event_date.nunique(),'timing_violations':violations})
timing=pd.DataFrame(timing); timing.to_parquet(RESULTS/'timing_audit.parquet',index=False)
display(timing); assert timing.timing_violations.eq(0).all()


## 4. Future-return interaction regressions

We estimate firm abnormal return as an event fixed effect plus gamma times historical beta times the current surprise. Event fixed effects absorb the common reaction to each announcement. Standard errors are two-way clustered by event and firm. A stable target implies positive gamma.

In [ ]:
interaction_rows=[]
for name,panel in panels.items():
    outcome=specifications[name]['outcome']
    for sample in ['all','development','validation','final_test']:
        subset=panel if sample=='all' else panel[panel.time_sample.eq(sample)]
        result=ftv.interaction_regression(subset,outcome,'prediction')
        result.update(specification=name,time_sample=sample)
        interaction_rows.append(result)
interaction=pd.DataFrame(interaction_rows)
interaction.to_parquet(RESULTS/'future_interaction_regressions.parquet',index=False)
display(interaction.sort_values(['specification','time_sample']))


## 5. Event-level correlations and predetermined portfolios

Event-level correlations prevent large cross-sections from hiding instability across announcements. Separately, firms are sorted into historical-beta quintiles before each quarter. The high-minus-low return spread should move positively with the surprise.

In [ ]:
correlation_frames=[]; portfolio_rows=[]
for name,panel in panels.items():
    outcome=specifications[name]['outcome']
    correlations=ftv.event_correlations(panel,outcome,'prediction'); correlations['specification']=name
    correlations['time_sample']=np.select([correlations.event_date.dt.year.le(2016),correlations.event_date.dt.year.le(2020)],['development','validation'],default='final_test')
    correlation_frames.append(correlations)
    work=panel.copy(); work['beta_percentile']=work.groupby('anchor_quarter').beta_winsorized.rank(method='average',pct=True)
    for event_date,group in work.groupby('event_date'):
        low=group.loc[group.beta_percentile.le(.20),outcome].mean(); high=group.loc[group.beta_percentile.gt(.80),outcome].mean(); shock=group.shock_value.iloc[0]
        portfolio_rows.append({'specification':name,'event_date':event_date,'shock_value':shock,'high_minus_low':high-low,'sign_adjusted_spread':(high-low)*np.sign(shock),'firms':len(group)})
event_correlations=pd.concat(correlation_frames,ignore_index=True)
portfolios=pd.DataFrame(portfolio_rows); portfolios['time_sample']=np.select([portfolios.event_date.dt.year.le(2016),portfolios.event_date.dt.year.le(2020)],['development','validation'],default='final_test')
correlation_summary=[]; portfolio_summary=[]
for (name,sample),group in event_correlations.groupby(['specification','time_sample']):
    test=stats.ttest_1samp(group.pearson.dropna(),0)
    correlation_summary.append({'specification':name,'time_sample':sample,'events':len(group),'mean_pearson':group.pearson.mean(),'median_pearson':group.pearson.median(),'positive_pearson_share':group.pearson.gt(0).mean(),'mean_spearman':group.spearman.mean(),'pearson_mean_t':test.statistic,'pearson_mean_p_two_sided':test.pvalue})
for (name,sample),group in portfolios.groupby(['specification','time_sample']):
    regression=stats.linregress(group.shock_value,group.high_minus_low); test=stats.ttest_1samp(group.sign_adjusted_spread.dropna(),0,alternative='greater')
    portfolio_summary.append({'specification':name,'time_sample':sample,'events':len(group),'spread_on_shock_slope':regression.slope,'slope_p_two_sided':regression.pvalue,'mean_sign_adjusted_spread':group.sign_adjusted_spread.mean(),'positive_sign_adjusted_share':group.sign_adjusted_spread.gt(0).mean(),'sign_adjusted_p_one_sided':test.pvalue})
correlation_summary=pd.DataFrame(correlation_summary); portfolio_summary=pd.DataFrame(portfolio_summary)
event_correlations.to_parquet(RESULTS/'event_level_prediction_correlations.parquet',index=False); portfolios.to_parquet(RESULTS/'future_beta_portfolios_by_event.parquet',index=False)
correlation_summary.to_parquet(RESULTS/'event_correlation_summary.parquet',index=False); portfolio_summary.to_parquet(RESULTS/'future_beta_portfolio_summary.parquet',index=False)
print('Event-level correlations'); display(correlation_summary)
print('Predetermined beta portfolios'); display(portfolio_summary)


## 6. Within-quarter shuffled-beta placebo

Primary historical betas are randomly reassigned among firms within each quarter. This preserves the shock sequence, return panel, and beta distribution while destroying firm identity. The upper-tail permutation p-value uses 200 frozen repetitions.

In [ ]:
primary=panels['stmt_ar00']
observed_slope,permutation_p,placebo=ftv.shuffled_slopes(primary,'abnormal_return_00','shock_stmt','beta_winsorized',repetitions=200,seed=20260902)
placebo.to_parquet(RESULTS/'primary_shuffled_beta_placebo_200.parquet',index=False)
placebo_summary=pd.DataFrame([{'observed_interaction_slope':observed_slope,'placebo_repetitions':len(placebo),'placebo_mean':placebo.placebo_slope.mean(),'placebo_standard_deviation':placebo.placebo_slope.std(),'placebo_95th_percentile':placebo.placebo_slope.quantile(.95),'upper_tail_permutation_p':permutation_p}])
placebo_summary.to_parquet(RESULTS/'primary_placebo_summary.parquet',index=False)
display(placebo_summary)


## 7. Measurement-validity gate

The target advances only if its interaction is positive overall and in the final test, its clustered t-statistic exceeds 1.645, its average event correlation is positive, and shuffled labels are rejected at 10%. These requirements were fixed before viewing any embedding result.

In [ ]:
primary_all=interaction.query("specification=='stmt_ar00' and time_sample=='all'").iloc[0]
primary_test=interaction.query("specification=='stmt_ar00' and time_sample=='final_test'").iloc[0]
primary_corr=event_correlations[event_correlations.specification.eq('stmt_ar00')]
validation_gate=pd.DataFrame([
 {'test':'primary interaction slope positive','value':primary_all.interaction_slope,'passed':primary_all.interaction_slope>0},
 {'test':'primary two-way clustered t > 1.645','value':primary_all.t_statistic,'passed':primary_all.t_statistic>1.645},
 {'test':'final-test interaction slope positive','value':primary_test.interaction_slope,'passed':primary_test.interaction_slope>0},
 {'test':'mean event-level Pearson positive','value':primary_corr.pearson.mean(),'passed':primary_corr.pearson.mean()>0},
 {'test':'shuffled-beta upper-tail p <= 0.10','value':permutation_p,'passed':permutation_p<=.10},
])
validation_gate.to_parquet(RESULTS/'measurement_validity_gate.parquet',index=False)
display(validation_gate)
print('STOP: inspect this gate before fitting any embedding-to-rate-sensitivity model.')
